In [ ]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [ ]:
# Model confi and generic function for getting the answer for a prompt
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

# Cache model outputs during iteration so re-runs do not call all providers every time.
# This keeps development/testing faster and avoids unnecessary API costs.
NOTEBOOK_DIR = Path("1_foundations/community_contributions/misi")
RESULTS_CACHE_PATH = (
    NOTEBOOK_DIR / "2_lab2_excercise.results_cache.json"
    if NOTEBOOK_DIR.exists()
    else Path("2_lab2_excercise.results_cache.json")
)


def load_cache(cache_path=RESULTS_CACHE_PATH):
    if not cache_path.exists():
        return {}
    try:
        with cache_path.open("r", encoding="utf-8") as f:
            cache_data = json.load(f)
        return cache_data if isinstance(cache_data, dict) else {}
    except Exception:
        return {}


def save_cache(data, cache_path=RESULTS_CACHE_PATH):
    cache_data = load_cache(cache_path)
    cache_data.update(data)
    with cache_path.open("w", encoding="utf-8") as f:
        json.dump(cache_data, f, indent=2)


MODEL_CONFIGS = {
    "question_raiser": "gpt-5-mini",
    "judge": "gpt-5-mini",
    "competitor_models": {
        "openai": {
            "provider": "openai",
            "model": "gpt-5-nano",
        },
        "anthropic": {
            "provider": "anthropic",
            "model": "claude-sonnet-4-5",
            "max_tokens": 1000,
        },
        "gemini": {
            "provider": "openai-compatible",
            "model": "gemini-2.5-flash",
            "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
            "api_key": google_api_key,
        },
        "deepseek": {
            "provider": "openai-compatible",
            "model": "deepseek-chat",
            "base_url": "https://api.deepseek.com/v1",
            "api_key": deepseek_api_key,
        },
        "ollama": {
            "provider": "openai-compatible",
            "model": "llama3.2",
            "base_url": "http://localhost:11434/v1",
            "api_key": "ollama",
        },
    },
}


def _anthropic_messages(prompt):
    return [{"role": "user", "content": prompt}]


def generate_answer(prompt, model_cfg):
    if isinstance(model_cfg, str):
        model_cfg = {"provider": "openai", "model": model_cfg}

    provider = model_cfg["provider"]

    if provider == "anthropic":
        client = Anthropic(api_key=anthropic_api_key)
        response = client.messages.create(
            model=model_cfg["model"],
            messages=_anthropic_messages(prompt),
            max_tokens=model_cfg.get("max_tokens", 1000),
        )
        return response.content[0].text

    client = OpenAI(
        api_key=model_cfg.get("api_key"),
        base_url=model_cfg.get("base_url"),
    )
    response = client.chat.completions.create(
        model=model_cfg["model"],
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [ ]:
# The challenging question

request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."

In [ ]:
cache_data = load_cache()
cached_question = cache_data.get("question")

# Reuse cached question so cached competitor answers stay relevant to the same prompt.
if cached_question:
    question = cached_question
    print(f"Loaded question from cache: {RESULTS_CACHE_PATH}")
else:
    question = generate_answer(request, MODEL_CONFIGS["question_raiser"])
    save_cache({"question_request": request, "question": question})
    print(f"Generated and cached question: {RESULTS_CACHE_PATH}")

# TODO - enable it again
# print(question)


In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:

def run_all_models(prompt, model_configs, cache_path=RESULTS_CACHE_PATH):
    # During development/testing, prefer cached results for repeatability and speed.
    cache_data = load_cache(cache_path)
    cached_results = cache_data.get("results")
    cached_question = cache_data.get("question")
    # Use cache only when all configured competitors are present and question matches.
    # This avoids returning answers from a different question.
    if (
        isinstance(cached_results, dict)
        and cached_question == prompt
        and all(name in cached_results for name in model_configs)
    ):
        return cached_results

    results = {}
    with ThreadPoolExecutor(max_workers=len(model_configs)) as executor:
        futures = {
            name: executor.submit(generate_answer, prompt, cfg)
            for name, cfg in model_configs.items()
        }
        for name, future in futures.items():
            try:
                results[name] = future.result()
            except Exception as e:
                results[name] = f"ERROR: {e}"

    # Persist latest run so the next notebook run can skip expensive model calls.
    save_cache({"question": prompt, "results": results}, cache_path)

    return results


results = run_all_models(question, MODEL_CONFIGS["competitor_models"])

for name, cfg in MODEL_CONFIGS["competitor_models"].items():
    model_name = cfg["model"]
    answer = results[name]
    # TODO: enable it again
    # display(Markdown(f"### {model_name}\n{answer}"))
    competitors.append(model_name)
    answers.append(answer)

In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank (1st is the best) and score them 1-100  in order of best to worst.
Give a score 1-100 to the quality of the answer, eg 100 is the perfect quality, it cannot be inproved, while 0 is totally unacceptable.
Also give a recommendation what needs to be added or changed or removed from the response to improve the score.
Respond with JSON, and only JSON, with the following format:
{{"descision": [{{"rank":"1","competitor_number":3,"score":98, "recommendation":"elaborate point 4 better,remove the contradiction..."}}, {{"rank":"2","competitor_number":2,"score":70, "recommendation":"the last sentence does not make sense, remove it"}}, {{"rank":"3","competitor_number":1,"score":22, "recommendation":"add a more details to the ..."}}, ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

In [92]:
# Judgement time!

descision = generate_answer(judge, MODEL_CONFIGS["judge"])
print(descision)


{"descision":[{"rank":"1","competitor_number":1,"score":95,"recommendation":"Very strong, complete, and practical answer: formal objective, constraints, pseudocode, belief updates, safety, metrics, templates, and sensitivity analysis. To improve: provide a compact formal mathematical statement of the optimization (variables, constraints, objective) suitable for an MILP, include a worked small numeric example or simulation showing runtime and solution quality, calibrate the survival‑benefit parameters from literature (or show how you would), and add more precise complexity bounds for the chosen heuristics (e.g., ALNS configuration and expected runtime/optimality gap). Also expand the redundancy policy into explicit optimization constraints or a formal multi‑objective tradeoff model."},{"rank":"2","competitor_number":3,"score":92,"recommendation":"Comprehensive and well‑reasoned: clear operational strategy, formal objective, probabilistic models with concrete Bayesian update example, saf

In [ ]:
# OK let's turn this into descision!

descision_dict = json.loads(descision)
ranks_and_scores = descision_dict["descision"]
for index, result in enumerate(ranks_and_scores):
    print(f"{result.get("rank")}: {competitors[result.get("competitor_number")-1]} with score: {result.get("score")}, recommendation is: {result.get("recommendation")}")
